# Top-K & K-Way Merge

Two heap patterns from the same toolbox. **Top-K** keeps only the k best elements in a **size-k heap** — O(n log k), beating a full O(n log n) sort when k ≪ n. **K-way merge** walks k sorted sequences at once through a **heap of their frontiers** — O(N log k). Both lean on the `heapq` from the heaps notebook.

> signal → template → worked examples → toolkit

**Contents**

1. **The signal**
2. **Top-K** — a heap of size k
3. **K-way merge** — a heap of frontiers
4. **When to use & toolkit**
5. **Complexity cheat-sheet**

## 1. The signal — a bounded heap, not a full sort

Two recognizable shapes, both solved by a **bounded** heap rather than sorting everything:

- **"the k largest / smallest / most-frequent"** — you don't need everything in order, just the top k. Keep a heap of **size k**: O(n log k) time, O(k) memory.
- **"merge k sorted lists" / "kth smallest in a sorted matrix" / "smallest range across k lists"** — you're consuming several sorted streams in order. Keep one **frontier** element per stream in a heap of size k and always advance the smallest.

The shared idea: a heap surfaces the current min/max in O(log n) without sorting the whole input.

## 2. Top-K — a heap of size k

To find the **k largest**, keep a **min-heap of size k**. Push each element; whenever the heap exceeds k, pop the smallest. What survives is the k largest, and the heap's root is the **k-th largest**. It's O(n log k) — and since k is usually small, far cheaper than sorting everything (O(n log n)).

(Counter-intuitive but the key: you use a *min*-heap to track the *largest* elements, because the smallest of your current top-k is exactly the one to evict next.)

In [1]:
import heapq

def k_largest(nums, k):
    heap = []                            # a MIN-heap holding the k largest seen so far
    for x in nums:
        heapq.heappush(heap, x)
        if len(heap) > k:
            heapq.heappop(heap)          # evict the smallest -> only the k largest remain
    return sorted(heap, reverse=True)

nums = [3, 1, 5, 12, 2, 11]
print("k_largest(nums, 3):", k_largest(nums, 3))

# top-K by frequency: count, then heap the (value, count) pairs by count
from collections import Counter
def top_k_frequent(nums, k):
    counts = Counter(nums)
    return [v for v, _ in heapq.nlargest(k, counts.items(), key=lambda kv: kv[1])]

print("top_k_frequent([1,1,1,2,2,3], 2):", top_k_frequent([1, 1, 1, 2, 2, 3], 2))

# the stdlib runs the size-k heap for you:
print("heapq.nlargest(3) :", heapq.nlargest(3, nums))
print("heapq.nsmallest(3):", heapq.nsmallest(3, nums))


k_largest(nums, 3): [12, 11, 5]
top_k_frequent([1,1,1,2,2,3], 2): [1, 2]
heapq.nlargest(3) : [12, 11, 5]
heapq.nsmallest(3): [1, 2, 3]


`heapq.nlargest(k, it)` / `nsmallest(k, it)` are the batteries-included version of exactly this size-k heap (same O(n log k)), and `Counter(...).most_common(k)` is the cleanest top-k-by-frequency. Hand-roll the heap only when elements arrive as a **stream** and you must maintain the top-k **online**.

## 3. K-way merge — a heap of frontiers

To merge k sorted lists you only ever need to compare their **current front elements**. Put one frontier per list into a heap of size k; repeatedly pop the smallest, output it, and push the **next** element from that same list. Total cost **O(N log k)** for N elements across k lists — versus O(N log N) if you concatenated and sorted.

In [2]:
import heapq

def merge_k_sorted(lists):
    heap = []
    for i, lst in enumerate(lists):
        if lst:
            heapq.heappush(heap, (lst[0], i, 0))     # (value, which list, index in that list)
    out = []
    while heap:
        val, i, j = heapq.heappop(heap)
        out.append(val)
        if j + 1 < len(lists[i]):                    # push that list's next element
            heapq.heappush(heap, (lists[i][j + 1], i, j + 1))
    return out

print("merge_k_sorted:", merge_k_sorted([[1, 4, 7], [2, 5, 8], [3, 6, 9]]))

# the stdlib merges sorted inputs lazily, in O(N log k), without materializing them:
print("heapq.merge   :", list(heapq.merge([1, 4, 7], [2, 5, 8], [3, 6, 9])))


merge_k_sorted: [1, 2, 3, 4, 5, 6, 7, 8, 9]
heapq.merge   : [1, 2, 3, 4, 5, 6, 7, 8, 9]


The tuple `(value, i, j)` carries the bookkeeping, and the trailing index `j` also breaks ties deterministically so `heapq` never has to compare the list payloads themselves. The same frontier-heap idea solves "kth smallest in a sorted matrix" (rows are the sorted lists) and "smallest range covering k lists.

## 4. When to use & toolkit

| The problem asks for… | Use |
|---|---|
| k largest / smallest | size-k heap (`heapq.nlargest` / `nsmallest`) |
| k most frequent | `Counter` + size-k heap (`most_common`) |
| kth largest / smallest | size-k heap; the answer is the root |
| merge k sorted lists / streams | frontier heap (`heapq.merge`) |
| kth smallest in a sorted matrix | frontier heap over the rows |

**Python toolkit:** `heapq.nlargest` / `nsmallest(k, it, key=...)` for one-shot top-k; `heapq.merge(*iterables, key=...)` for lazy k-way merge (memory-light — it never materializes the inputs); `Counter(...).most_common(k)` for frequency. Hand-roll a heap only for **online / streaming** top-k.

**Why size-k, not sort:** when **k ≪ n**, O(n log k) beats O(n log n) — and for a stream you can't sort at all. If you genuinely need *everything* in order, just sort.

## 5. Complexity cheat-sheet

| Task | Approach | Time | Space |
|---|---|:---:|:---:|
| k largest / smallest | size-k heap | O(n log k) | O(k) |
| top-k frequent | `Counter` + heap | O(n log k) | O(n) |
| merge k sorted (N total) | frontier heap | O(N log k) | O(k) |
| full sort *(for contrast)* | Timsort | O(n log n) | O(n) |

<sub>The win is the **bound**: keeping only k things in the heap (top-k) or only k frontiers (merge) caps the `log` factor at `log k` instead of `log n`.</sub>